# Analyzing Experiment Results

This notebook shows how to:
- Load experiment results
- Compute statistics
- Create visualizations
- Generate publication-ready figures

## Setup

Import required libraries:

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from agentick.visualization.experiment_plots import ExperimentPlotter

# Set plotting style
plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

## Run a Quick Experiment

First, let's run a quick experiment to generate some data:

In [ ]:
from agentick.experiments import ExperimentConfig, ExperimentRunner

# Create a simple experiment config
config = ExperimentConfig(
    name="notebook_demo",
    agent={"type": "random"},
    tasks=["GoToGoal-v0", "MazeNavigation-v0"],
    difficulties=["easy"],
    n_episodes=10,
    n_seeds=3,
    render_modes=["ascii"],
    output_dir="results/notebook_demo",
)

# Run experiment
print("Running experiment...")
runner = ExperimentRunner(config)
results = runner.run()
print(f"\nExperiment complete! Results saved to: {config.output_dir}")

## Loading Results

Load the experiment results from disk:

In [ ]:
# Find the latest experiment directory
results_dir = Path("results/notebook_demo")
experiment_dirs = sorted(results_dir.glob("*"))
latest_exp = experiment_dirs[-1] if experiment_dirs else None

print(f"Loading results from: {latest_exp}")

# Load summary
with open(latest_exp / "summary.json") as f:
    summary = json.load(f)

print("\nSummary Statistics:")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

## Per-Task Analysis

Analyze results for each task:

In [ ]:
# Load per-task results
task_dirs = list((latest_exp / "per_task").glob("*"))

task_results = []
for task_dir in task_dirs:
    metrics_file = task_dir / "metrics.json"
    if metrics_file.exists():
        with open(metrics_file) as f:
            data = json.load(f)
            task_results.append(
                {
                    "task": task_dir.name,
                    "mean_return": data["aggregate_metrics"]["mean_return"],
                    "success_rate": data["aggregate_metrics"]["success_rate"],
                    "mean_length": data["aggregate_metrics"]["mean_length"],
                }
            )

# Create DataFrame
df = pd.DataFrame(task_results)
print("\nPer-Task Results:")
print(df.to_string(index=False))

## Visualization

Create visualizations of the results:

In [ ]:
# Bar chart of success rates
fig, ax = plt.subplots(figsize=(10, 6))

tasks = df["task"].tolist()
success_rates = df["success_rate"].tolist()

ax.bar(tasks, success_rates, color="steelblue", alpha=0.8)
ax.set_xlabel("Task", fontsize=12)
ax.set_ylabel("Success Rate", fontsize=12)
ax.set_title("Success Rate by Task", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Return distribution
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(tasks))
width = 0.35

ax.bar(x, df["mean_return"], width, label="Mean Return", color="coral", alpha=0.8)
ax.set_xlabel("Task", fontsize=12)
ax.set_ylabel("Mean Return", fontsize=12)
ax.set_title("Mean Return by Task", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(tasks, rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Episode-Level Analysis

Look at individual episode trajectories:

In [ ]:
# Load episodes for first task
first_task_dir = task_dirs[0]
episode_files = sorted((first_task_dir / "episodes").glob("*.json"))[:10]

episode_data = []
for ep_file in episode_files:
    with open(ep_file) as f:
        ep = json.load(f)
        episode_data.append(
            {
                "episode": ep_file.stem,
                "length": ep["episode_length"],
                "reward": ep["total_reward"],
                "success": ep["success"],
            }
        )

ep_df = pd.DataFrame(episode_data)
print(f"\nFirst 10 Episodes of {first_task_dir.name}:")
print(ep_df.to_string(index=False))

In [ ]:
# Episode length distribution
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(ep_df["length"], bins=20, color="mediumseagreen", alpha=0.7, edgecolor="black")
ax.axvline(
    ep_df["length"].mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {ep_df['length'].mean():.1f}",
)
ax.set_xlabel("Episode Length", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title(f"Episode Length Distribution - {first_task_dir.name}", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## Using ExperimentPlotter

Use the built-in plotter for comprehensive visualizations:

In [ ]:
# Create plotter
plotter = ExperimentPlotter(str(latest_exp))

# Generate all plots
output_dir = "results/plots"
plotter.plot_all(output_dir=output_dir)

print(f"\nGenerated plots saved to: {output_dir}")
print("\nGenerated files:")
for plot_file in sorted(Path(output_dir).glob("*.png")):
    print(f"  - {plot_file.name}")

## Statistical Summary

Compute comprehensive statistics:

In [ ]:
# Compute statistics across all tasks
print("\nOverall Statistics:")
print(f"  Total tasks: {len(task_results)}")
print(f"  Mean success rate: {df['success_rate'].mean():.2%}")
print(f"  Std success rate: {df['success_rate'].std():.2%}")
print(f"  Mean return: {df['mean_return'].mean():.3f}")
print(f"  Mean episode length: {df['mean_length'].mean():.1f}")
print(f"\nBest performing task: {df.loc[df['success_rate'].idxmax(), 'task']}")
print(f"  Success rate: {df['success_rate'].max():.2%}")
print(f"\nWorst performing task: {df.loc[df['success_rate'].idxmin(), 'task']}")
print(f"  Success rate: {df['success_rate'].min():.2%}")

## Export Results

Export analysis to CSV for further processing:

In [ ]:
# Export to CSV
csv_path = latest_exp / "analysis.csv"
df.to_csv(csv_path, index=False)
print(f"\nResults exported to: {csv_path}")

# Also save episode-level data
ep_csv_path = latest_exp / "episodes_analysis.csv"
ep_df.to_csv(ep_csv_path, index=False)
print(f"Episode data exported to: {ep_csv_path}")

## Next Steps

- **03_compare_agents.ipynb** - Compare multiple agents
- **04_leaderboard_analysis.ipynb** - Analyze leaderboard submissions
- **examples/plotting/** - More visualization examples
- Run larger experiments with more tasks and agents!